# Lab 3 - Authentication & Access Control

## Aim

- Explore what makes a **strong password**.
- Implement password hashing using **bcrypt** with an application-level pepper.
- Add a second factor using **TOTP** (time-based one-time passwords).

This notebook combines explanation and small experiments.


## Password Strength Function

We evaluate a password based on:

- Length (8+ and 12+ characters).
- Use of lowercase, uppercase, digits, and special characters.
- Approximate entropy in bits: `entropy ≈ length * log2(pool_size)`.
- Whether the password is in a small set of obviously bad choices.

This is **not** a full production password-strength meter,
but it illustrates the reasoning process.


In [1]:
import math
import string

BAD_PASSWORDS = {"password", "123456", "qwerty", "letmein"}

def password_score(pw: str) -> dict:
    score = 0
    pool = 0

    if len(pw) >= 8:
        score += 1
    if len(pw) >= 12:
        score += 1

    sets = {
        "lower": any(c.islower() for c in pw),
        "upper": any(c.isupper() for c in pw),
        "digits": any(c.isdigit() for c in pw),
        "special": any(c in string.punctuation for c in pw),
    }

    if sets["lower"]: pool += 26
    if sets["upper"]: pool += 26
    if sets["digits"]: pool += 10
    if sets["special"]: pool += len(string.punctuation)

    score += sum(sets.values())
    entropy = len(pw) * math.log2(pool or 1)
    common = pw.lower() in BAD_PASSWORDS

    return {"score": score, "entropy_bits": entropy, "is_common": common}

# Quick demo:
for pw in ["password", "Tr0ub4dor&3", "correct horse battery staple"]:
    print(pw, "->", password_score(pw))


password -> {'score': 2, 'entropy_bits': 37.60351774512874, 'is_common': True}
Tr0ub4dor&3 -> {'score': 5, 'entropy_bits': 72.10047736845401, 'is_common': False}
correct horse battery staple -> {'score': 3, 'entropy_bits': 131.6123121079506, 'is_common': False}


## Hashing Passwords with bcrypt, Salt, and Pepper

- **Salt**: generated and stored per password by `bcrypt.gensalt()`.
- **Pepper**: secret stored at the application level (not in the database).
- `bcrypt` is intentionally slow, which makes brute-force attacks harder.


In [2]:
import bcrypt

APP_PEPPER = b"super-secret-pepper"

def hash_password(pw: str) -> bytes:
    salted = APP_PEPPER + pw.encode()
    return bcrypt.hashpw(salted, bcrypt.gensalt())

def verify_password(pw: str, stored: bytes) -> bool:
    return bcrypt.checkpw(APP_PEPPER + pw.encode(), stored)

# Demo of hashing and verification
stored_hash = hash_password("ExamplePass123!")
print("Stored hash:", stored_hash)

print("Verify correct password:", verify_password("ExamplePass123!", stored_hash))
print("Verify wrong password:", verify_password("wrong", stored_hash))

Stored hash: b'$2b$12$mA96JwskaZFUtHXMm.3pPeHnaPjMwCBkUgn0EfwjfFxBXyM6zaBUi'
Verify correct password: True
Verify wrong password: False


## Adding TOTP-Based 2FA

We add an extra factor using time-based one-time passwords:

- Generate a TOTP secret for the user.
- The user stores it in an authenticator app.
- On login, we check both the password **and** a 6-digit TOTP code.


In [3]:
import pyotp

def demo_login_flow():
    print("=== Demo registration ===")
    password = "ExamplePass123!"
    pw_hash = hash_password(password)

    # In a real system, the secret would be per user
    totp_secret = pyotp.random_base32()
    print("TOTP secret:", totp_secret)

    # Make it easier for the lab: show the current TOTP code
    totp = pyotp.TOTP(totp_secret)
    current_code = totp.now()
    print("Current TOTP code for demo purposes:", current_code)

    print("\n=== Demo login ===")
    attempt = input("Password: ")

    if not verify_password(attempt, pw_hash):
        print("Password incorrect.")
        return

    code = input("Enter 6-digit TOTP code: ")

    if totp.verify(code):
        print("Login successful with 2FA.")
    else:
        print("2FA code incorrect.")

demo_login_flow()


=== Demo registration ===
TOTP secret: YFKEXU5V2UOKUI45HAM543IATCWFXX5B
Current TOTP code for demo purposes: 707545

=== Demo login ===


Password:  ExamplePass123!
Enter 6-digit TOTP code:  707545


Login successful with 2FA.
